# 02 - Matriz de retencion mensual por cohorte

Este notebook construye y explora la matriz de retencion mensual por cohorte a partir de la base procesada [cohortes__retencion_mensual__v1.parquet](../data/processed/cohortes__retencion_mensual__v1.parquet).

Dentro del caso, este notebook sirvio para:

- visualizar la estructura de cohortes del proyecto;
- revisar la retencion mensual global;
- explorar diferencias por pais usando `primary_country`;
- dejar una base interpretable para el notebook de hallazgos y para la narrativa ejecutiva del dashboard.


## Lectura metodologica

La base de cohortes contiene dos alcances:

- `all_countries`: cohortes agregadas sobre todo el dataset multicountry.
- `primary_country`: cohortes segmentadas por el pais principal del cliente.

Importante: los clientes multicountry no se duplican entre paises. Para la segmentacion por pais se asignan a `primary_country`, lo que mantiene una granularidad estable a nivel cliente.


In [18]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 200)

ROOT_DIR = Path.cwd().resolve()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

COHORT_PATH = ROOT_DIR / "data" / "processed" / "cohortes__retencion_mensual__v1.parquet"

if not COHORT_PATH.exists():
    raise FileNotFoundError(
        "No se encontro la base de cohortes. Ejecuta primero `python -m src.process_cohorts`."
    )

cohort_df = pd.read_parquet(COHORT_PATH)
cohort_df["cohort_scope"] = cohort_df["cohort_scope"].astype("string")
cohort_df["country"] = cohort_df["country"].astype("string")
cohort_df["cohort_month"] = cohort_df["cohort_month"].astype("string")
cohort_df["activity_month"] = cohort_df["activity_month"].astype("string")

cohort_df.head()


,cohort_scope,country,cohort_month,activity_month,cohort_index,cohort_size,cohort_multicountry_customers,n_customers_active,n_multicountry_customers_active,retention_rate,n_orders,n_lines,total_quantity,total_revenue_gbp,avg_revenue_per_active_customer_gbp,avg_orders_per_active_customer,avg_lines_per_active_customer
0,all_countries,ALL_COUNTRIES,2009-12,2009-12,0,955,3,955,3,1.0,1512,30272,398660,683504.01,715.711005,1.583246,31.698429
1,all_countries,ALL_COUNTRIES,2009-12,2010-01,1,955,3,337,1,0.35288,569,11901,277892,394723.981,1171.287777,1.688427,35.31454
2,all_countries,ALL_COUNTRIES,2009-12,2010-02,2,955,3,319,1,0.334031,564,11451,247565,295931.572,927.685179,1.768025,35.896552
3,all_countries,ALL_COUNTRIES,2009-12,2010-03,3,955,3,406,2,0.425131,717,14228,322261,378663.12,932.667783,1.76601,35.044335
4,all_countries,ALL_COUNTRIES,2009-12,2010-04,4,955,3,363,2,0.380105,629,12634,172017,305901.29,842.703278,1.732782,34.804408


In [19]:
resumen_general = pd.Series(
    {
        "filas_totales": len(cohort_df),
        "scopes_disponibles": ", ".join(sorted(cohort_df["cohort_scope"].dropna().unique().tolist())),
        "cohort_month_min": cohort_df["cohort_month"].min(),
        "cohort_month_max": cohort_df["cohort_month"].max(),
        "activity_month_max": cohort_df["activity_month"].max(),
        "cohort_index_max": int(cohort_df["cohort_index"].max()),
        "paises_en_scope_primary_country": int(cohort_df.loc[cohort_df["cohort_scope"] == "primary_country", "country"].nunique()),
    },
    name="valor",
).to_frame()
resumen_general


,valor
filas_totales,4844
scopes_disponibles,"all_countries, primary_country"
cohort_month_min,2009-12
cohort_month_max,2011-12
activity_month_max,2011-12
cohort_index_max,24
paises_en_scope_primary_country,41


In [20]:
def build_retention_matrix(df: pd.DataFrame) -> pd.DataFrame:
    matrix = (
        df.pivot_table(
            index="cohort_month",
            columns="cohort_index",
            values="retention_rate",
            aggfunc="first",
        )
        .sort_index()
        .sort_index(axis=1)
    )
    matrix.columns.name = "mes_desde_primera_compra"
    return matrix


def build_customer_matrix(df: pd.DataFrame) -> pd.DataFrame:
    matrix = (
        df.pivot_table(
            index="cohort_month",
            columns="cohort_index",
            values="n_customers_active",
            aggfunc="first",
        )
        .sort_index()
        .sort_index(axis=1)
    )
    matrix.columns.name = "mes_desde_primera_compra"
    return matrix


def style_retention_cell(value: float) -> str:
    if pd.isna(value):
        return "background-color: #f5f5f5; color: #9ca3af;"
    if value >= 0.50:
        return "background-color: #1d4ed8; color: white;"
    if value >= 0.35:
        return "background-color: #60a5fa; color: #111827;"
    if value >= 0.20:
        return "background-color: #bfdbfe; color: #111827;"
    if value >= 0.10:
        return "background-color: #dbeafe; color: #111827;"
    return "background-color: #eff6ff; color: #6b7280;"


def format_retention_matrix(matrix: pd.DataFrame):
    return (
        matrix.style.format("{:.1%}")
        .map(style_retention_cell)
        .set_caption("Matriz de retencion mensual")
    )


def get_scope_df(scope: str, country: str | None = None) -> pd.DataFrame:
    df = cohort_df.loc[cohort_df["cohort_scope"] == scope].copy()
    if country is not None:
        df = df.loc[df["country"] == country].copy()
    return df.sort_values(["cohort_month", "cohort_index"])


## Cohortes globales (`all_countries`)

Esta vista resume la retencion del dataset completo y permite observar el comportamiento agregado antes de abrir la lectura por pais.


In [21]:
global_df = get_scope_df("all_countries")

global_cohort_summary = (
    global_df.loc[global_df["cohort_index"] == 0, ["cohort_month", "cohort_size", "cohort_multicountry_customers"]]
    .sort_values("cohort_month")
    .reset_index(drop=True)
)
global_cohort_summary


,cohort_month,cohort_size,cohort_multicountry_customers
0,2009-12,955,3
1,2010-01,383,2
2,2010-02,374,1
3,2010-03,443,0
4,2010-04,294,0
5,2010-05,254,0
6,2010-06,270,1
7,2010-07,186,0
8,2010-08,162,1
9,2010-09,243,2


In [22]:
global_retention_matrix = build_retention_matrix(global_df)
display(format_retention_matrix(global_retention_matrix))


mes_desde_primera_compra,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24
cohort_month,,,,,,,,,,,,,,,,,,,,,,,,,
2009-12,100.0%,35.3%,33.4%,42.5%,38.0%,35.9%,37.7%,34.2%,33.6%,36.2%,42.2%,49.5%,37.6%,28.3%,24.4%,30.3%,26.3%,30.3%,28.3%,26.0%,25.5%,31.5%,30.5%,40.7%,19.7%
2010-01,100.0%,20.6%,31.1%,30.5%,26.4%,30.0%,25.8%,23.0%,27.9%,31.9%,30.3%,17.2%,22.2%,17.8%,18.8%,15.1%,23.5%,19.8%,18.5%,19.6%,24.3%,19.3%,24.5%,5.7%,
2010-02,100.0%,23.8%,22.5%,29.1%,24.6%,20.1%,19.3%,28.6%,25.4%,27.5%,11.5%,12.6%,15.2%,17.4%,12.3%,20.1%,16.0%,16.3%,14.4%,23.0%,23.0%,16.3%,5.9%,,
2010-03,100.0%,19.0%,23.0%,24.2%,23.3%,20.3%,24.6%,30.2%,27.5%,10.8%,11.5%,14.2%,20.1%,16.3%,20.1%,16.9%,17.4%,15.6%,17.6%,20.1%,21.2%,7.9%,,,
2010-04,100.0%,19.4%,19.4%,16.3%,18.4%,22.4%,27.6%,26.2%,10.5%,10.9%,7.5%,13.9%,13.9%,15.6%,15.6%,15.6%,13.9%,15.0%,18.0%,22.4%,5.8%,,,,
2010-05,100.0%,15.7%,16.9%,17.3%,17.7%,25.6%,21.3%,12.6%,5.9%,8.3%,11.4%,13.4%,15.4%,15.4%,9.8%,12.6%,13.8%,16.5%,15.4%,4.7%,,,,,
2010-06,100.0%,17.4%,18.9%,20.4%,23.0%,28.5%,12.6%,8.9%,8.1%,11.9%,10.7%,13.7%,14.8%,12.2%,11.1%,12.2%,13.3%,20.4%,5.2%,,,,,,
2010-07,100.0%,15.6%,18.3%,29.6%,29.0%,14.0%,11.3%,14.5%,14.5%,11.3%,13.4%,14.5%,13.4%,13.4%,19.4%,17.2%,23.7%,8.1%,,,,,,,
2010-08,100.0%,20.4%,29.6%,32.1%,17.3%,11.7%,9.9%,12.3%,13.6%,13.0%,13.0%,12.3%,15.4%,18.5%,17.9%,19.8%,6.8%,,,,,,,,


In [23]:
global_active_customers_matrix = build_customer_matrix(global_df)
global_active_customers_matrix


mes_desde_primera_compra,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24
cohort_month,,,,,,,,,,,,,,,,,,,,,,,,,
2009-12,955,337,319,406,363,343,360,327,321,346,403,473,359,270,233,289,251,289,270,248,244,301,291,389,188
2010-01,383,79,119,117,101,115,99,88,107,122,116,66,85,68,72,58,90,76,71,75,93,74,94,22,<NA>
2010-02,374,89,84,109,92,75,72,107,95,103,43,47,57,65,46,75,60,61,54,86,86,61,22,<NA>,<NA>
2010-03,443,84,102,107,103,90,109,134,122,48,51,63,89,72,89,75,77,69,78,89,94,35,<NA>,<NA>,<NA>
2010-04,294,57,57,48,54,66,81,77,31,32,22,41,41,46,46,46,41,44,53,66,17,<NA>,<NA>,<NA>,<NA>
2010-05,254,40,43,44,45,65,54,32,15,21,29,34,39,39,25,32,35,42,39,12,<NA>,<NA>,<NA>,<NA>,<NA>
2010-06,270,47,51,55,62,77,34,24,22,32,29,37,40,33,30,33,36,55,14,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2010-07,186,29,34,55,54,26,21,27,27,21,25,27,25,25,36,32,44,15,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2010-08,162,33,48,52,28,19,16,20,22,21,21,20,25,30,29,32,11,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## Cohortes por pais (`primary_country`)

La siguiente seccion permite revisar la matriz de retencion para un pais especifico usando la asignacion `primary_country` definida en la base cliente. Esta apertura complementa la lectura global sin romper la granularidad a nivel cliente.


In [24]:
top_countries = (
    cohort_df.loc[cohort_df["cohort_scope"] == "primary_country", ["country", "cohort_month", "cohort_size"]]
    .drop_duplicates()
    .groupby("country", as_index=False)["cohort_size"]
    .sum()
    .sort_values("cohort_size", ascending=False)
    .reset_index(drop=True)
)
top_countries.head(10)


,country,cohort_size
0,United Kingdom,5350
1,Germany,106
2,France,95
3,Spain,38
4,Belgium,28
5,Portugal,24
6,Netherlands,22
7,Switzerland,22
8,Sweden,19
9,Italy,17


In [25]:
selected_country = "Germany"
selected_country


'Germany'

In [26]:
country_df = get_scope_df("primary_country", selected_country)

country_cohort_summary = (
    country_df.loc[country_df["cohort_index"] == 0, ["country", "cohort_month", "cohort_size", "cohort_multicountry_customers"]]
    .sort_values("cohort_month")
    .reset_index(drop=True)
)
country_cohort_summary


,country,cohort_month,cohort_size,cohort_multicountry_customers
0,Germany,2009-12,10,0
1,Germany,2010-01,9,0
2,Germany,2010-02,10,0
3,Germany,2010-03,5,0
4,Germany,2010-04,7,0
5,Germany,2010-05,4,0
6,Germany,2010-06,3,0
7,Germany,2010-07,8,0
8,Germany,2010-08,1,0
9,Germany,2010-09,2,0


In [27]:
country_retention_matrix = build_retention_matrix(country_df)
display(format_retention_matrix(country_retention_matrix))


mes_desde_primera_compra,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24
cohort_month,,,,,,,,,,,,,,,,,,,,,,,,,
2009-12,100.0%,40.0%,60.0%,40.0%,80.0%,50.0%,60.0%,40.0%,50.0%,50.0%,60.0%,70.0%,60.0%,60.0%,30.0%,70.0%,40.0%,60.0%,50.0%,60.0%,40.0%,60.0%,60.0%,60.0%,10.0%
2010-01,100.0%,22.2%,55.6%,33.3%,33.3%,22.2%,22.2%,22.2%,22.2%,55.6%,33.3%,22.2%,22.2%,0.0%,33.3%,11.1%,22.2%,11.1%,22.2%,33.3%,22.2%,0.0%,33.3%,0.0%,
2010-02,100.0%,50.0%,30.0%,10.0%,30.0%,30.0%,30.0%,60.0%,60.0%,30.0%,10.0%,60.0%,40.0%,40.0%,10.0%,50.0%,50.0%,20.0%,30.0%,20.0%,40.0%,60.0%,10.0%,,
2010-03,100.0%,20.0%,0.0%,20.0%,20.0%,20.0%,20.0%,20.0%,40.0%,0.0%,20.0%,20.0%,0.0%,60.0%,0.0%,0.0%,0.0%,40.0%,40.0%,40.0%,20.0%,0.0%,,,
2010-04,100.0%,14.3%,14.3%,28.6%,14.3%,28.6%,42.9%,28.6%,28.6%,28.6%,14.3%,28.6%,14.3%,28.6%,42.9%,14.3%,14.3%,14.3%,28.6%,42.9%,14.3%,,,,
2010-05,100.0%,0.0%,25.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,25.0%,50.0%,0.0%,0.0%,25.0%,25.0%,25.0%,0.0%,25.0%,0.0%,,,,,
2010-06,100.0%,0.0%,33.3%,33.3%,33.3%,33.3%,33.3%,33.3%,33.3%,0.0%,33.3%,33.3%,0.0%,66.7%,0.0%,100.0%,33.3%,33.3%,0.0%,,,,,,
2010-07,100.0%,25.0%,25.0%,37.5%,50.0%,25.0%,50.0%,37.5%,37.5%,25.0%,62.5%,12.5%,37.5%,37.5%,37.5%,37.5%,37.5%,12.5%,,,,,,,
2010-08,100.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,100.0%,0.0%,0.0%,,,,,,,,


## Guia de lectura

- Cada fila representa una cohorte definida por el mes de primera compra.
- La columna `0` siempre debe mostrar retencion de `100%`, porque corresponde al mes de ingreso de la cohorte.
- La columna `1` representa la retencion al primer mes posterior.
- Las cohortes mas recientes tienen menos columnas observables porque aun no han tenido suficiente tiempo de seguimiento.
- En la segmentacion por pais, la interpretacion debe considerar tamano de cohorte: tasas altas en cohortes pequenas pueden ser menos estables.


## Aporte al analisis del caso

Este notebook no busca cerrar por si solo la interpretacion de negocio. Su valor dentro del caso es dejar visible la estructura de retencion sobre la que despues se construyen hallazgos comparables, por ejemplo:

- retencion del mes 1, 3 y 6;
- comparacion entre cohortes tempranas y recientes;
- comparacion por pais para los mercados con mayor volumen.
